<a href="https://colab.research.google.com/github/longvujulix/Analysis-Project/blob/main/WebScraping_Selenium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Portfolio] Web Scraping: HP Laptop Market Data Extraction

## 1. Mục đích (Objectives)
Dự án này được thực hiện nhằm xây dựng một quy trình tự động hóa việc thu thập dữ liệu sản phẩm từ website thương mại điện tử. Mục tiêu cụ thể bao gồm:
*   **Thu thập dữ liệu thực tế:** Lấy thông tin chi tiết về dòng laptop HP (tên, giá, thông số, tình trạng hàng) để phục vụ cho việc phân tích thị trường.
*   **Xây dựng Dataset:** Chuyển đổi dữ liệu thô từ cấu trúc HTML không cấu trúc thành tập dữ liệu có cấu trúc (Structured Data) để sẵn sàng cho các bước xử lý dữ liệu (Data Cleaning) và phân tích (Exploratory Data Analysis - EDA) sau này.
*   **Chứng minh kỹ năng:** Thể hiện khả năng sử dụng Python trong việc tương tác với Web, xử lý các cấu trúc DOM phức tạp và quản lý dữ liệu.

## 2. Mô tả Website (Target Website)
*   **Tên Website:** TNC Store
*   **URL mục tiêu:** [tnc.com.vn/laptop-hp-chinh-hang.html](https://www.tnc.com.vn/laptop-hp-chinh-hang.html)
*   **Đặc điểm dữ liệu:** Website chứa thông tin đa dạng về các dòng Laptop HP từ văn phòng đến cao cấp. Dữ liệu được tổ chức theo dạng lưới (grid), yêu cầu kỹ thuật bóc tách chính xác các thẻ HTML để tránh lấy nhầm thông tin của các sản phẩm quảng cáo hoặc các thành phần trang web khác.

## 3. Kỹ thuật sử dụng (Tech Stack & Techniques)
Dự án áp dụng các thư viện và kỹ thuật phổ biến trong lĩnh vực Data Analysis:
*   **Ngôn ngữ:** Python.
*   **Thư viện Scraping:**
    *   `Requests`: Gửi các truy vấn HTTP GET để lấy mã nguồn HTML từ Server.
    *   `BeautifulSoup4`: Phân tích cú pháp (parsing) HTML và duyệt qua các cây thư mục DOM để trích xuất dữ liệu mục tiêu.
*   **Xử lý dữ liệu:**
    *   `Pandas`: Chuyển đổi dữ liệu sau khi cào được vào DataFrame, thực hiện định dạng lại kiểu dữ liệu (ví dụ: chuyển giá từ dạng chuỗi "20.000.000đ" sang dạng số nguyên) và xuất file CSV/Excel.
*   **Kỹ thuật bổ trợ:**
    *   Xử lý phân trang (Pagination handling) để lấy toàn bộ danh sách sản phẩm.
    *   Sử dụng `User-Agent` để giả lập trình duyệt, giúp quy trình cào dữ liệu ổn định và chuyên nghiệp hơn.

---

**Người thực hiện:** Trần Long Vũ  
**Vị trí hướng tới:** Business Data Analyst / CX Analyst

# Extract Data

In [ ]:
# Import các thư viện cần thiết
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [ ]:
# Khởi tạo danh sách lưu trữ dữ liệu tổng
all_product_names = []
all_product_prices = []
all_detail_links = []
all_image_links = []
all_image_filenames = []

In [ ]:
# 1. Khởi tạo trình duyệt và truy cập trang đích
options = webdriver.ChromeOptions()
options.add_argument('headless') # Không hiện cửa sổ trình duyệt
driver = webdriver.Chrome(options=options)

In [ ]:
url = 'https://www.tnc.com.vn/laptop-hp-chinh-hang.html'
driver.get(url)
time.sleep(5)

In [ ]:
# Cào các thông tin về sản phẩm ở trang Product Listings
i = 1
while True:
    # Tạo URL mới dựa trên biến i
    current_url = f'https://www.tnc.com.vn/laptop-hp-chinh-hang.html?p={i}'
    print(f"Đang cào dữ liệu tại trang: {i}...")

    driver.get(current_url)
    time.sleep(3)

    products = driver.find_elements(By.XPATH, '//div[@class="cr-product-card"]')

    if len(products) == 0:
        print(f"Đã hết sản phẩm tại trang {i}. Kết thúc vòng lặp.")
        break

    for product in products:
        try:
            # Lấy tên
            details = product.find_element(By.XPATH, './/div[@class="cr-product-details"]')
            name = details.find_element(By.XPATH, './/h3').text.strip()
            link = details.find_element(By.XPATH, './/a').get_attribute("href")

            # Lấy giá
            price = product.find_element(By.XPATH, './/span[@class="new-price"]').text.strip()

            # Lấy ảnh
            img_element = product.find_element(By.XPATH, './/div[@class="cr-product-image"]//img')
            img_src = img_element.get_attribute("data-src") or img_element.get_attribute("src")

            # Lưu vào danh sách
            all_product_names.append(name)
            all_detail_links.append(link)
            all_product_prices.append(price)
            all_image_links.append(img_src)

        except:
            continue

    i += 1

    if i > 50:
        break

print(f"Hoàn thành! Tổng cộng thu thập được {len(all_product_names)} sản phẩm.")

In [ ]:
# Cào các thông tin đánh giá và chi tiết của sản phẩm ở trang từng trang Product
all_product_codes = []
all_specs = []
all_ratings = []

for link in all_detail_links:
    driver.get(link)
    time.sleep(2)

    # Khởi tạo giá trị mặc định cho mỗi sản phẩm
    current_code = "N/A"
    current_specs = "N/A"
    current_rating = "0/5"

    try:
        # 1. Xác định khung cha chứa toàn bộ thông tin
        container = driver.find_element(By.CLASS_NAME, "cr-size-and-weight")

        # --- LẤY MÃ SẢN PHẨM (Thẻ p ngang hàng với Star và List) ---
        try:
            current_code = container.find_element(By.XPATH, './/p').text.replace("Mã:", "").strip()
        except:
            pass

        # --- LẤY RATING STAR (Đếm icon ri-star-fill) ---
        try:
            # Truy cập sâu vào: cr-review-star -> cr-star -> đếm các thẻ i có class fill
            star_fills = container.find_elements(By.XPATH, './/div[@class="cr-review-star"]//div[@class="cr-star"]//i[contains(@class, "ri-star-fill")]')
            current_rating = f"{len(star_fills)}/5"
        except:
            current_rating = "0/5"

        try:
            # Dùng XPath tìm thẻ li bên trong container
            spec_elements = container.find_elements(By.XPATH, './/li')
            if spec_elements:
                # Nối tất cả các dòng li lại thành một chuỗi văn bản
                current_specs = " | ".join([s.text.strip() for s in spec_elements])
            else:
                # Nếu không phải li, thử tìm trực tiếp văn bản trong khối list
                current_specs = container.find_element(By.XPATH, './/*[contains(@class, "list")]').text.strip()
        except:
            pass

    except Exception as e:
        print(f"Lỗi truy xuất tại link {link}: {e}")

    # Lưu vào danh sách tổng
    all_product_codes.append(current_code)
    all_specs.append(current_specs)
    all_ratings.append(current_rating)

    print(f"Xong: {current_code} | Star: {current_rating} | Specs: {current_specs[:30]}...")

driver.quit()

In [ ]:
# Clean data và lưu dữ liệu
import pandas as pd
import numpy as np

# 1. Chuẩn hóa cột giá bán (numeric_prices)
# Đảm bảo các giá trị trong all_product_prices đã được loại bỏ 'đ', '.', ',' và ép kiểu float
cleaned_prices = [str(x).replace('đ', '').replace('.', '').replace(',', '').strip() for x in all_product_prices]

# Chuyển sang kiểu số, nếu không phải số thì để là NaN
numeric_prices = []
for x in cleaned_prices:
    try:
        numeric_prices.append(float(x))
    except:
        numeric_prices.append(np.nan)

# 2. Tạo DataFrame từ các danh sách đã thu thập
df_products = pd.DataFrame({
    "Mã sản phẩm": all_product_codes,
    "Tên sản phẩm": all_product_names,
    "Giá bán": numeric_prices,
    "Thông số kỹ thuật": all_specs,
    "Đánh giá người dùng": all_ratings,
    "Tên tập tin hình": all_image_filenames
})

# 3. Kiểm tra kiểu dữ liệu
print(df_products.dtypes)

# 4. Lưu dữ liệu vào file Products.csv
df_products.to_csv("Products.csv", index=False, encoding="utf-8-sig")

print(f"Đã lưu thành công {len(df_products)} sản phẩm vào file Products.csv!")

# Hiển thị 5 dòng đầu tiên để kiểm tra
print(df_products.head())